In [3]:
import torch
torch.backends.cudnn.benchmark = True

In [4]:
# Use GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Currently using: {device}")

if device == "cuda":
   print(torch.cuda.get_device_name(0))
else:
   print("No GPU available, using CPU.")

Currently using: cuda
NVIDIA GeForce RTX 4060 Laptop GPU


In [12]:
import pandas as pd
import os

In [42]:
def load_and_save_data():
   # Load the data
   df_corpus = pd.read_parquet("hf://datasets/carlfeynman/Bharat_NanoMSMARCO_ml/corpus/train-00000-of-00001.parquet", engine='pyarrow')
   df_qrels = pd.read_parquet("hf://datasets/carlfeynman/Bharat_NanoMSMARCO_ml/qrels/train-00000-of-00001.parquet", engine='pyarrow')
   df_queries = pd.read_parquet("hf://datasets/carlfeynman/Bharat_NanoMSMARCO_ml/queries/train-00000-of-00001.parquet",engine='pyarrow')

   print(os.getcwd())
   # Save as JSONL
   df_corpus.to_json("../datasets/Bharat_NanoMSMARCO/corpus.jsonl", orient="records", lines=True)
   df_queries.to_json("../datasets/Bharat_NanoMSMARCO/queries.jsonl", orient="records", lines=True)
   df_qrels.to_csv("../datasets/Bharat_NanoMSMARCO/qrels.tsv", sep='\t', index=False)


In [43]:
load_and_save_data()

/home/nakul/devfiles/PROJECTS/malayalam-ir-bench/notebooks


In [29]:
from sentence_transformers import SentenceTransformer, util
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

model = SentenceTransformer("intfloat/multilingual-e5-base", device=device)

Using: cuda


In [45]:
import json

# Load corpus (documents)
with open("../datasets/Bharat_NanoMSMARCO/corpus.jsonl", "r", encoding="utf-8") as f:
    docs = [json.loads(line) for line in f]
    doc_texts = [doc["text"] for doc in docs]
    doc_ids = [doc["_id"] for doc in docs]

# Load queries
with open("../datasets/Bharat_NanoMSMARCO/queries.jsonl", "r", encoding="utf-8") as f:
    queries = [json.loads(line) for line in f]
    query_texts = [q["text"] for q in queries]
    query_ids = [q["_id"] for q in queries]

# Load qrels (relevance judgments)
df_qrels = pd.read_csv("../datasets/Bharat_NanoMSMARCO/qrels.tsv", sep='\t')
qrels = {}
for _, row in df_qrels.iterrows():
    qid = row['query-id']
    did = row['corpus-id']
    qrels.setdefault(qid, {})[did] = 1



In [39]:
# Move model to GPU explicitly (in case it's not already)
model.to(device)

# Encode documents and queries (use normalize_embeddings for cosine)
doc_emb = model.encode(doc_texts, convert_to_tensor=True, device=device, normalize_embeddings=True)
query_emb = model.encode(query_texts, convert_to_tensor=True, device=device, normalize_embeddings=True)


In [40]:
scores = util.dot_score(query_emb, doc_emb)  # Shape: (num_queries, num_docs)

In [41]:
import numpy as np

def dcg_at_k(scores, k):
    return scores[0] + sum(score / np.log2(i + 2) for i, score in enumerate(scores[1:k]))

def ndcg_at_k(true_rels, ranked_ids, k):
    rels = [true_rels.get(doc_ids[i], 0) for i in ranked_ids[:k]]
    ideal = sorted(true_rels.values(), reverse=True)[:k]
    idcg = dcg_at_k(ideal, k)
    return dcg_at_k(rels, k) / idcg if idcg > 0 else 0.0

def recall_at_k(true_ids, ranked_ids, k):
    return len(set(true_ids) & set([doc_ids[i] for i in ranked_ids[:k]])) / len(true_ids)

# Evaluate
K = 5
all_recalls = []
all_ndcgs = []

for i, qid in enumerate(query_ids):
    relevant_docs = qrels.get(qid, {})
    if not relevant_docs: continue

    ranked = torch.topk(scores[i], k=K+10).indices.cpu().tolist()  # Get top results
    recall = recall_at_k(list(relevant_docs.keys()), ranked, K)
    ndcg = ndcg_at_k(relevant_docs, ranked, K)

    all_recalls.append(recall)
    all_ndcgs.append(ndcg)

print(f"📈 Recall@{K}: {np.mean(all_recalls):.3f}")
print(f"📈 NDCG@{K}: {np.mean(all_ndcgs):.3f}")


📈 Recall@5: 0.520
📈 NDCG@5: 0.443


In [ ]:
from pytrec_eval import RelevanceEvaluator

evaluator = RelevanceEvaluator(qrels, )